# The Road Ahead

Tire brand (Michelin, Goodyear, Bridgestone, Continental) comparison for the future of vehicles! Electrical vehicle and autonomous vehicle analysis using open source data (web-scraping google news, wikipedia and brand websites).

Check the README and associated slides for more information.



## 1. Install dependencies

In [ ]:
!pip install -q \
    gradio \
    plotly \
    pandas \
    requests \
    beautifulsoup4 \
    feedparser \
    lxml \
    langchain \
    langchain-openai \
    langchain-core \
    langgraph \
    python-dotenv


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.6/98.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 20.1 MB/s eta 0:00:00


## 2. Imports and configuration

In [ ]:
import os
import pathlib

import gradio as gr
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# limited and monitored key from Michelin
os.environ["OPENAI_API_KEY"] = "API-KEY-HERE"
pathlib.Path("data").mkdir(exist_ok=True)


## 3. Web scraping

In [ ]:
%%writefile scraper.py
"""
scraper.py
Webscrapes the following sources for data collection:
1. GOOGLE NEWS RSS
   URL: https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en
   # feedparser parses the RSS feed, no auth required
   # collects article title, snippet, published date, source name

2. WIKIPEDIA MEDIAWIKI API
   URL: https://en.wikipedia.org/w/api.php?action=query&prop=extracts&explaintext=1
   # GET request to the MediaWiki REST API
   # collects full article plaintext filtered to EV/AV-relevant paragraphs

3. BRAND NEWSROOMS
   # requests + BeautifulSoup scraping from each brand press page
   # collects headline text from recent press releases
   Brand URLs:
     Michelin    -> https://www.michelin.com/en/press-releases/
     Goodyear    -> https://corporate.goodyear.com/us/en/media/goodyear-news.html
     Bridgestone -> https://www.bridgestoneamericas.com/en/newsroom/press-releases?year=all
     Continental -> https://www.continental.com/en/press/press-releases/
"""

import json
import time
from datetime import datetime
from pathlib import Path
import feedparser
import requests
from bs4 import BeautifulSoup

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; RoadAheadBot/1.0; tire-brand-research-tool)"
}

BRANDS = ["Michelin", "Goodyear", "Bridgestone", "Continental"]

EV_AV_KEYWORDS = [
    "electric", "EV", "autonomous", "self-driving", "AV",
    "connected", "sensor", "rolling resistance", "sustainable",
    "carbon", "ADAS", "automated", "battery",
]

NEWSROOMS = {
    "Michelin":    "https://www.michelin.com/en/press-releases/",
    "Goodyear":    "https://corporate.goodyear.com/us/en/media/goodyear-news.html",
    "Bridgestone": "https://www.bridgestoneamericas.com/en/newsroom/press-releases?year=all",
    "Continental": "https://www.continental.com/en/press/press-releases/",
}

def scrape_google_news(brand: str, topic: str, max_articles: int = 15) -> list[dict]:
    query = f"{brand} {topic}".replace(" ", "+")
    url = f"https://news.google.com/rss/search?q={query}&hl=en-US&gl=US&ceid=US:en"
    try:
        feed = feedparser.parse(url)
        articles = []
        for entry in feed.entries[:max_articles]:
            articles.append({
                "title":     entry.get("title", ""),
                "snippet":   BeautifulSoup(entry.get("summary", ""), "lxml").get_text()[:300],
                "published": entry.get("published", ""),
                "source":    entry.get("source", {}).get("title", "Google News"),
                "link":      entry.get("link", ""),
            })
        return articles
    except Exception as e:
        print(f"Google News error for {brand}/{topic}: {e}")
        return []

def scrape_wikipedia(brand: str) -> str:
    WIKI_TITLES = {
        "Michelin":    "Michelin",
        "Goodyear":    "Goodyear Tire and Rubber Company",
        "Bridgestone": "Bridgestone",
        "Continental": "Continental AG",
    }
    title = WIKI_TITLES.get(brand, brand)
    params = {
        "action":      "query",
        "prop":        "extracts",
        "titles":      title,
        "explaintext": 1,
        "format":      "json",
    }
    try:
        resp = requests.get(
            "https://en.wikipedia.org/w/api.php",
            params=params, headers=HEADERS, timeout=10
        )
        pages = resp.json()["query"]["pages"]
        text = next(iter(pages.values())).get("extract", "")
        relevant = [
            para.strip() for para in text.split("\n")
            if any(kw.lower() in para.lower() for kw in EV_AV_KEYWORDS)
        ]
        return " ".join(relevant[:30])
    except Exception as e:
        print(f"Wikipedia error for {brand}: {e}")
        return ""

def scrape_newsroom(brand: str, max_headlines: int = 20) -> list[str]:
    url = NEWSROOMS.get(brand)
    if not url:
        return []
    try:
        resp = requests.get(url, headers=HEADERS, timeout=12)
        soup = BeautifulSoup(resp.text, "lxml")
        candidates = [
            tag.get_text(strip=True)
            for tag in soup.find_all(["h2", "h3", "h4", "a"])
            if 10 < len(tag.get_text(strip=True)) < 200
        ]
        return candidates[:max_headlines]
    except Exception as e:
        print(f"Newsroom error for {brand}: {e}")
        return []

def collect_all(force_refresh: bool = False) -> dict:
    cache_path = DATA_DIR / "raw_data.json"

    if cache_path.exists() and not force_refresh:
        print("Loading from cache")
        with open(cache_path) as f:
            return json.load(f)

    print("Starting data collection")
    results = {}

    for brand in BRANDS:
        print(f"  {brand}")
        ev_news  = scrape_google_news(brand, "electric vehicle tire", 12)
        time.sleep(0.5)
        av_news  = scrape_google_news(brand, "autonomous vehicle technology", 12)
        time.sleep(0.5)
        wiki     = scrape_wikipedia(brand)
        time.sleep(0.3)
        newsroom = scrape_newsroom(brand)
        time.sleep(0.5)

        results[brand] = {
            "ev_news":      ev_news,
            "av_news":      av_news,
            "wikipedia":    wiki,
            "newsroom":     newsroom,
            "collected_at": datetime.utcnow().isoformat(),
        }
        print(f"    ev_news={len(ev_news)} av_news={len(av_news)} newsroom={len(newsroom)}")

    with open(cache_path, "w") as f:
        json.dump(results, f, indent=2)

    print(f"Data saved to {cache_path}")
    return results

Overwriting scraper.py


## 4. LLM scoring and agent

In [ ]:
%%writefile analyzer.py
"""
analyzer.py
Scores each brand on five EV/AV dimensions using GPT-4o-mini
and builds a LangGraph ReAct agent for the chat interface
"""

import json
from pathlib import Path
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

DATA_DIR = Path("data")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

DIMENSIONS = {
    "ev_tire_innovation": (
        "Evidence of EV-specific tire products, low rolling resistance technology, "
        "higher load-rating innovations for heavy EV batteries, or noise-reduction "
        "solutions for the quieter EV drivetrain."
    ),
    "av_integration": (
        "Sensor-embedded or connected tire systems, tire-to-vehicle data feeds, "
        "ADAS partnerships, or direct collaboration with autonomous vehicle "
        "platforms such as Waymo, Tesla, or GM Cruise."
    ),
    "sustainability_commitment": (
        "Concrete carbon reduction targets, use of recycled or bio-based materials, "
        "circular economy programs, net-zero timelines, and biodiversity commitments."
    ),
    "strategic_partnerships": (
        "Joint ventures or OEM agreements with EV manufacturers, investment in "
        "EV/AV startups, university research partnerships, or government program "
        "participation related to future mobility."
    ),
    "innovation_pipeline": (
        "Announced R&D programs, patent filings in EV/AV domains, innovation labs, "
        "accelerator programs, or public commitments to future mobility R&D spend."
    ),
}

def build_brand_context(brand: str, raw_data: dict) -> str:
    data = raw_data.get(brand, {})
    news_text = "\n".join(
        a.get("title", "") + ". " + a.get("snippet", "")
        for a in data.get("ev_news", []) + data.get("av_news", [])
    )
    newsroom_text = "\n".join(data.get("newsroom", [])[:15])
    wikipedia_text = data.get("wikipedia", "")[:3000]
    return (
        f"=== Recent News ===\n{news_text}\n\n"
        f"=== Press Releases ===\n{newsroom_text}\n\n"
        f"=== Wikipedia ===\n{wikipedia_text}"
    )

def score_brand(brand: str, raw_data: dict) -> dict:
    context = build_brand_context(brand, raw_data)
    dim_descriptions = "\n".join(f'- "{k}": {v}' for k, v in DIMENSIONS.items())

    prompt = f"""You are a strategic analyst specializing in the tire industry's transition
to electric and autonomous vehicles.

Score {brand} from 0 to 100 on each dimension below based only on the provided data.
Score low when evidence is thin.

Dimensions:
{dim_descriptions}

Return ONLY valid JSON with this structure (no markdown fences):
{{
  "brand": "{brand}",
  "scores": {{
    "ev_tire_innovation": <int 0-100>,
    "av_integration": <int 0-100>,
    "sustainability_commitment": <int 0-100>,
    "strategic_partnerships": <int 0-100>,
    "innovation_pipeline": <int 0-100>
  }},
  "justifications": {{
    "ev_tire_innovation": "<1-2 sentence summary>",
    "av_integration": "<1-2 sentence summary>",
    "sustainability_commitment": "<1-2 sentence summary>",
    "strategic_partnerships": "<1-2 sentence summary>",
    "innovation_pipeline": "<1-2 sentence summary>"
  }},
  "overall_summary": "<3-4 sentence competitive positioning summary>"
}}

Data:
{context[:6000]}
"""
    try:
        response = llm.invoke(prompt)
        raw = response.content.strip()
        if raw.startswith("```"):
            raw = raw.split("```")[1]
            if raw.startswith("json"):
                raw = raw[4:]
        return json.loads(raw)
    except Exception as e:
        print(f"Scoring error for {brand}: {e}")
        return {
            "brand": brand,
            "scores": {k: 0 for k in DIMENSIONS},
            "justifications": {k: "Scoring failed." for k in DIMENSIONS},
            "overall_summary": "Scoring failed.",
        }

def build_scores(raw_data: dict, force_refresh: bool = False) -> dict:
    cache_path = DATA_DIR / "scores.json"

    if cache_path.exists() and not force_refresh:
        with open(cache_path) as f:
            return json.load(f)

    print("Scoring brands...")
    scores = {}
    for brand in raw_data:
        print(f"  {brand}")
        scores[brand] = score_brand(brand, raw_data)

    with open(cache_path, "w") as f:
        json.dump(scores, f, indent=2)

    print("Scores saved")
    return scores

def build_agent(scores: dict, raw_data: dict):

    @tool
    def get_brand_scores(brand: str) -> str:
        """
        Get EV/AV scores and justifications for one brand.
        Brand must be: Michelin, Goodyear, Bridgestone, or Continental.
        """
        return json.dumps(scores.get(brand, {"error": "not found"}), indent=2)

    @tool
    def compare_all_scores(dimension: str) -> str:
        """
        Compare all four brands on one dimension.
        Dimension must be one of: ev_tire_innovation, av_integration,
        sustainability_commitment, strategic_partnerships, innovation_pipeline.
        """
        results = {
            brand: {
                "score": data.get("scores", {}).get(dimension),
                "justification": data.get("justifications", {}).get(dimension, "")
            }
            for brand, data in scores.items()
        }
        ranked = sorted(results.items(), key=lambda x: x[1]["score"] or 0, reverse=True)
        return json.dumps(dict(ranked), indent=2)

    @tool
    def get_recent_news(brand: str) -> str:
        """
        Get recent EV and AV news headlines for one brand.
        Brand must be: Michelin, Goodyear, Bridgestone, or Continental.
        """
        data = raw_data.get(brand, {})
        headlines = [
            {"title": a.get("title"), "source": a.get("source"), "date": a.get("published")}
            for a in (data.get("ev_news", []) + data.get("av_news", []))[:10]
        ]
        return json.dumps(headlines, indent=2)

    @tool
    def get_overall_rankings(_: str = "") -> str:
        """Get all four brands ranked by average EV/AV score. No input needed."""
        ranked = {
            brand: {
                "average_score": round(
                    sum(data.get("scores", {}).values()) / len(data.get("scores", {})), 1
                ) if data.get("scores") else 0,
                "scores": data.get("scores", {}),
                "summary": data.get("overall_summary", ""),
            }
            for brand, data in scores.items()
        }
        return json.dumps(
            dict(sorted(ranked.items(), key=lambda x: x[1]["average_score"], reverse=True)),
            indent=2
        )

    SYSTEM = """You are a mobility industry analyst covering the tire sector's transition
to electric and autonomous vehicles. You have scored competitive intelligence across
five dimensions for Michelin, Goodyear, Bridgestone, and Continental.

Be specific. Cite scores. Evaluate Michelin honestly against competitors."""

    return create_react_agent(
        llm,
        [get_brand_scores, compare_all_scores, get_recent_news, get_overall_rankings],
        prompt=SYSTEM
    )

Writing analyzer.py


## 5. Check data collection

In [ ]:
from analyzer import build_scores, build_agent
from scraper import BRANDS, collect_all

raw_data = collect_all()

rows = []
for brand in BRANDS:
    d = raw_data.get(brand, {})
    ev_articles  = d.get("ev_news", [])
    av_articles  = d.get("av_news", [])
    newsroom     = d.get("newsroom", [])
    wiki         = d.get("wikipedia", "")
    collected_at = d.get("collected_at", "unknown")
    rows.append({
        "Brand":              brand,
        "EV News Articles":   len(ev_articles),
        "AV News Articles":   len(av_articles),
        "Newsroom Headlines": len(newsroom),
        "Wikipedia (chars)":  len(wiki),
        "Total Text Inputs":  len(ev_articles) + len(av_articles) + len(newsroom) + (1 if wiki else 0),
        "Collected At (UTC)": collected_at[:19].replace("T", " "),
    })

summary_df = pd.DataFrame(rows)
display(summary_df)

total_ev   = sum(len(raw_data[b].get("ev_news",   [])) for b in BRANDS)
total_av   = sum(len(raw_data[b].get("av_news",   [])) for b in BRANDS)
total_nr   = sum(len(raw_data[b].get("newsroom",  [])) for b in BRANDS)
total_wiki = sum(len(raw_data[b].get("wikipedia", "")) for b in BRANDS)

totals_df = pd.DataFrame([{
    "Total EV Articles":        total_ev,
    "Total AV Articles":        total_av,
    "Total Newsroom Headlines": total_nr,
    "Total Wikipedia Chars":    total_wiki,
    "Total LLM Text Inputs":    total_ev + total_av + total_nr + 4,
}])
display(totals_df)

/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Starting data collection
  Michelin
    ev_news=12 av_news=12 newsroom=20
  Goodyear
    ev_news=12 av_news=12 newsroom=0
  Bridgestone
    ev_news=12 av_news=12 newsroom=0
  Continental
    ev_news=12 av_news=12 newsroom=20
Data saved to data/raw_data.json


,Brand,EV News Articles,AV News Articles,Newsroom Headlines,Wikipedia (chars),Total Text Inputs,Collected At (UTC)
0,Michelin,12,12,20,8295,45,2026-05-08 01:11:20
1,Goodyear,12,12,0,13088,25,2026-05-08 01:11:23
2,Bridgestone,12,12,0,13322,25,2026-05-08 01:11:26
3,Continental,12,12,20,7299,45,2026-05-08 01:11:30


,Total EV Articles,Total AV Articles,Total Newsroom Headlines,Total Wikipedia Chars,Total LLM Text Inputs
0,48,48,40,42004,140


## 6. Build app

In [ ]:
scores = build_scores(raw_data)
agent  = build_agent(scores, raw_data)

COLORS = {
    "Michelin":    "#005DAA",
    "Goodyear":    "#003087",
    "Bridgestone": "#E31837",
    "Continental": "#FF6600",
}
DIMENSION_LABELS = {
    "ev_tire_innovation":        "EV Tire Innovation",
    "av_integration":            "AV Integration",
    "sustainability_commitment": "Sustainability",
    "strategic_partnerships":    "Strategic Partnerships",
    "innovation_pipeline":       "Innovation Pipeline",
}
BRAND_EMOJI = {
    "Michelin": "🔵", "Goodyear": "🔷",
    "Bridgestone": "🔴", "Continental": "🟠",
}

def hex_to_rgba(hex_color: str, alpha: float = 0.2) -> str:
    h = hex_color.lstrip("#")
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f"rgba({r}, {g}, {b}, {alpha})"

def score_color(v):
    return "#27ae60" if v >= 70 else "#f39c12" if v >= 45 else "#e74c3c"

def make_radar(selected_brands):
    dim_keys   = list(DIMENSION_LABELS.keys())
    dim_labels = list(DIMENSION_LABELS.values())
    fig = go.Figure()
    for brand in selected_brands:
        s      = scores.get(brand, {}).get("scores", {})
        values = [s.get(d, 0) for d in dim_keys] + [s.get(dim_keys[0], 0)]
        fig.add_trace(go.Scatterpolar(
            r=values,
            theta=dim_labels + [dim_labels[0]],
            fill="toself",
            fillcolor=hex_to_rgba(COLORS[brand], alpha=0.2),
            line=dict(color=COLORS[brand], width=2.5),
            name=brand,
            hovertemplate="<b>" + brand + "</b><br>%{theta}: %{r}/100<extra></extra>",
        ))
    fig.update_layout(
        polar=dict(
            bgcolor="white",
            radialaxis=dict(
                visible=True, range=[0, 100],
                tickfont=dict(size=10, color="#888"),
                gridcolor="#e9ecef",
            ),
            angularaxis=dict(tickfont=dict(size=12, color="#333"), gridcolor="#e9ecef"),
        ),
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=-0.18, xanchor="center", x=0.5),
        paper_bgcolor="rgba(0,0,0,0)",
        height=500,
        margin=dict(t=40, b=80, l=60, r=60),
    )
    return fig

def make_bar(dimension, selected_brands):
    rows = [
        {"Brand": b, "Score": scores.get(b, {}).get("scores", {}).get(dimension, 0)}
        for b in selected_brands
    ]
    df = pd.DataFrame(rows).sort_values("Score", ascending=True)
    fig = px.bar(
        df, x="Score", y="Brand", orientation="h",
        color="Brand", color_discrete_map=COLORS,
        text="Score", range_x=[0, 100], height=240,
        title=f"{DIMENSION_LABELS.get(dimension, dimension)} — Brand Comparison",
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(
        showlegend=False,
        paper_bgcolor="rgba(0,0,0,0)",
        plot_bgcolor="rgba(0,0,0,0)",
        xaxis=dict(gridcolor="#e9ecef"),
        margin=dict(t=40, b=10),
    )
    return fig

def make_evidence_html(dimension, selected_brands):
    parts = []
    for brand in selected_brands:
        just  = scores.get(brand, {}).get("justifications", {}).get(dimension, "No data.")
        color = COLORS[brand]
        parts.append(
            f'<div style="border-left:4px solid {color}; padding:8px 14px; '
            f'margin-bottom:10px; background:white; border-radius:0 8px 8px 0; '
            f'box-shadow:0 1px 6px rgba(0,0,0,0.05)">'
            f'<strong style="color:{color}">{BRAND_EMOJI[brand]} {brand}</strong><br>'
            f'<span style="font-size:0.9rem; color:#444">{just}</span></div>'
        )
    return "".join(parts)

def make_scorecard_html(brand):
    data  = scores.get(brand, {})
    s     = data.get("scores", {})
    justs = data.get("justifications", {})
    avg   = round(sum(s.values()) / len(s), 1) if s else 0
    color = COLORS[brand]

    html = (
        f'<div style="background:#f8f9fa; border-radius:10px; padding:14px; '
        f'margin-bottom:16px; font-style:italic; color:#555">'
        f'{data.get("overall_summary", "")}</div>'
        f'<div style="font-size:0.85rem; color:#888; margin-bottom:14px">'
        f'Average score: <strong style="color:{color}">{avg}/100</strong></div>'
    )
    for dim_key, dim_label in DIMENSION_LABELS.items():
        val  = s.get(dim_key, 0)
        just = justs.get(dim_key, "")
        bc   = score_color(val)
        html += (
            f'<div style="margin-bottom:14px">'
            f'<div style="display:flex; justify-content:space-between; '
            f'align-items:center; margin-bottom:4px">'
            f'<span style="font-weight:600; font-size:0.9rem">{dim_label}</span>'
            f'<span style="background:{bc}; color:white; padding:2px 12px; '
            f'border-radius:20px; font-weight:700; font-size:0.85rem">{val}/100</span>'
            f'</div>'
            f'<div style="background:#f0f0f0; border-radius:8px; height:8px">'
            f'<div style="background:{bc}; width:{val}%; height:8px; border-radius:8px">'
            f'</div></div>'
            f'<div style="font-size:0.8rem; color:#777; margin-top:4px">{just}</div>'
            f'</div>'
        )
    return html

def make_news_html(selected_brands, topic_filter):
    parts = []
    for brand in selected_brands:
        brand_data = raw_data.get(brand, {})
        if topic_filter == "Electric Vehicle":
            articles = brand_data.get("ev_news", [])
        elif topic_filter == "Autonomous Vehicle":
            articles = brand_data.get("av_news", [])
        else:
            articles = brand_data.get("ev_news", []) + brand_data.get("av_news", [])

        color = COLORS[brand]
        parts.append(f"<h4 style='color:{color}'>{BRAND_EMOJI[brand]} {brand}</h4>")

        if not articles:
            parts.append("<p style='color:#aaa'>No articles found.</p>")
        else:
            for a in articles[:5]:
                snippet = a.get("snippet", "")
                parts.append(
                    f'<div style="background:white; border-radius:12px; padding:14px 18px; '
                    f'margin-bottom:10px; border-left:4px solid {color}; '
                    f'box-shadow:0 1px 8px rgba(0,0,0,0.05)">'
                    f'<div style="font-weight:600; font-size:0.95rem">'
                    f'<a href="{a.get("link", "#")}" target="_blank" '
                    f'style="color:#1a1a2e; text-decoration:none">{a.get("title", "Untitled")}</a>'
                    f'</div>'
                    f'<div style="font-size:0.82rem; color:#777; margin-top:4px">'
                    f'{snippet[:180]}{"..." if len(snippet) > 180 else ""}</div>'
                    f'<div style="font-size:0.75rem; color:#aaa; margin-top:6px">'
                    f'{a.get("source", "")} · {a.get("published", "")[:16]}</div>'
                    f'</div>'
                )
        parts.append("<br>")
    return "".join(parts)

def make_metric_tiles_html(selected_brands):
    tiles = []
    for brand in selected_brands:
        s   = scores.get(brand, {}).get("scores", {})
        avg = round(sum(s.values()) / len(s), 1) if s else 0
        color = COLORS[brand]
        tiles.append(
            f'<div style="background:white; border-radius:14px; padding:18px; '
            f'text-align:center; box-shadow:0 2px 12px rgba(0,0,0,0.07); '
            f'flex:1; min-width:120px">'
            f'<div style="font-size:2rem">{BRAND_EMOJI[brand]}</div>'
            f'<div style="font-weight:700; color:#1a1a2e; margin:4px 0">{brand}</div>'
            f'<div style="font-size:2rem; font-weight:800; color:{color}">{avg}</div>'
            f'<div style="font-size:0.75rem; color:#888">avg score / 100</div>'
            f'</div>'
        )
    return (
        '<div style="display:flex; gap:16px; flex-wrap:wrap; margin-bottom:8px">'
        + "".join(tiles)
        + '</div>'
    )

def chat_fn(message, history):
    try:
        result = agent.invoke({"messages": [{"role": "user", "content": message}]})
        return result["messages"][-1].content
    except Exception as e:
        return f"Agent error: {e}"

CSS = (
    ".gradio-container { background-color: #f4f6fb !important; } "
    ".tab-nav button { font-size: 0.95rem !important; }"
)

with gr.Blocks(title="The Road Ahead", theme=gr.themes.Soft(), css=CSS) as demo:

    gr.HTML(
        '<div style="background:linear-gradient(135deg,#1a1a2e,#16213e); '
        'border-radius:20px; padding:28px 32px; margin-bottom:16px">'
        '<h1 style="color:white; margin:0; font-size:1.9rem">The Road Ahead</h1>'
        '<p style="color:#ffffffbb; margin:8px 0 0">Comparing major tire brands on '
        'Electric Vehicle and Autonomous Vehicle positioning</p>'
        '<p style="color:#ffffff55; margin:6px 0 0; font-size:0.8rem">'
        'Sources: Google News RSS · Wikipedia MediaWiki API · Brand newsrooms · '
        'Powered by LangGraph + GPT-4o-mini</p></div>'
    )

    brand_select = gr.CheckboxGroup(choices=BRANDS, value=BRANDS, label="Brands to display")
    metric_tiles = gr.HTML(make_metric_tiles_html(BRANDS))
    brand_select.change(fn=make_metric_tiles_html, inputs=brand_select, outputs=metric_tiles)

    with gr.Tabs():

        with gr.Tab("Radar Dashboard"):
            gr.Markdown("### EV & AV Positioning")
            gr.Markdown(
                "*Scores generated by GPT-4o-mini from scraped news, "
                "Wikipedia, and brand press releases.*"
            )
            radar_plot = gr.Plot(make_radar(BRANDS))
            brand_select.change(fn=make_radar, inputs=brand_select, outputs=radar_plot)

            gr.Markdown("---\n### Drill down by dimension")
            dim_dropdown = gr.Dropdown(
                choices=list(DIMENSION_LABELS.keys()),
                value="ev_tire_innovation",
                label="Dimension",
                type="value",
            )
            bar_plot      = gr.Plot(make_bar("ev_tire_innovation", BRANDS))
            evidence_html = gr.HTML(make_evidence_html("ev_tire_innovation", BRANDS))

            def update_drill(dim, brands):
                return make_bar(dim, brands), make_evidence_html(dim, brands)

            for trigger in [dim_dropdown, brand_select]:
                trigger.change(
                    fn=update_drill,
                    inputs=[dim_dropdown, brand_select],
                    outputs=[bar_plot, evidence_html],
                )

        with gr.Tab("Brand Scorecards"):
            gr.Markdown("### Brand Scorecards")
            for brand in BRANDS:
                with gr.Accordion(f"{BRAND_EMOJI[brand]} {brand}", open=(brand == "Michelin")):
                    gr.HTML(make_scorecard_html(brand))

        with gr.Tab("News Feed"):
            gr.Markdown("### Recent EV & AV News by Brand")
            gr.Markdown("*Sourced from Google News RSS at time of last data refresh.*")
            topic_radio = gr.Radio(
                choices=["All", "Electric Vehicle", "Autonomous Vehicle"],
                value="All",
                label="Filter by topic",
            )
            news_html = gr.HTML(make_news_html(BRANDS, "All"))
            for trigger in [brand_select, topic_radio]:
                trigger.change(
                    fn=make_news_html,
                    inputs=[brand_select, topic_radio],
                    outputs=news_html,
                )

        with gr.Tab("AI Analyst"):
            gr.Markdown("### AI Analyst")
            gr.Markdown(
                "Ask anything about EV/AV positioning. "
                "The agent has access to all scores, justifications, and news."
            )
            gr.ChatInterface(
                fn=chat_fn,
                examples=[
                    "Which brand is best positioned for the EV transition overall?",
                    "How does Michelin compare to Bridgestone on AV integration?",
                    "Which brand has the weakest sustainability commitment and why?",
                    "Rank all four brands by strategic partnerships score.",
                    "What are the key EV tire innovation differences between Goodyear and Continental?",
                ],
                type="messages",
            )

print("App built")

Scoring brands...
  Michelin
  Goodyear
  Bridgestone
  Continental
Scores saved


/tmp/ipykernel_8218/1405266090.py:214: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="The Road Ahead", theme=gr.themes.Soft(), css=CSS) as demo:
/tmp/ipykernel_8218/1405266090.py:214: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="The Road Ahead", theme=gr.themes.Soft(), css=CSS) as demo:


App built


## 7. Launch

In [ ]:
demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ca79521d5291832cc9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
